# HKEX News Announcements - Step-by-Step Tutorial

This notebook walks through how to programmatically pull listed company announcements from the **Hong Kong Stock Exchange (HKEX)** using the undocumented JSON API.

## How It Works

The [HKEXnews website](https://www1.hkexnews.hk/listedco/listconews/index/lci.html?lang=en) loads announcement data from an internal JSON endpoint:

```
https://www1.hkexnews.hk/ncms/json/eds/lcisehk1relsdc_{page}.json
```

This is **not an official public API** — it was discovered by inspecting the website's network requests. It returns paginated JSON with a `newsInfoLst` array containing all recent announcements.

**No API key required.** Just HTTP GET requests.

---
## Step 1: Install Dependencies

Only `requests` is needed (standard HTTP library).

In [ ]:
!pip install requests -q

In [ ]:
import requests
import json
from datetime import datetime

print("Dependencies loaded successfully!")

---
## Step 2: Understand the API Endpoint

The JSON endpoint is paginated. Each page returns ~20 announcements.

| Parameter | Value |
|-----------|-------|
| Base URL | `https://www1.hkexnews.hk` |
| Endpoint | `/ncms/json/eds/lcisehk1relsdc_{page}.json` |
| Pages | 1, 2, 3, ... (1-indexed) |
| Auth | None required |
| Format | JSON |

Let's fetch page 1 and examine the raw response:

In [ ]:
# Define the endpoint
HKEX_ENDPOINT = "https://www1.hkexnews.hk/ncms/json/eds/lcisehk1relsdc_{page}.json"

# Set headers to mimic a browser request
headers = {
    "User-Agent": "Mozilla/5.0 (compatible; HKEXParser/1.0)",
    "Accept": "application/json",
    "Referer": "https://www1.hkexnews.hk/listedco/listconews/index/lci.html?lang=en",
}

# Fetch page 1
url = HKEX_ENDPOINT.format(page=1)
print(f"Fetching: {url}")

response = requests.get(url, headers=headers, timeout=30)
print(f"Status: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type', 'N/A')}")
print(f"Response size: {len(response.content)} bytes")

---
## Step 3: Examine the Raw JSON Response

Let's look at the top-level keys and the structure of the response:

In [ ]:
# Parse the JSON
data = response.json()

# Show top-level keys
print("Top-level keys:")
for key in data.keys():
    value = data[key]
    if isinstance(value, list):
        print(f"  '{key}': list with {len(value)} items")
    elif isinstance(value, dict):
        print(f"  '{key}': dict with keys {list(value.keys())}")
    else:
        print(f"  '{key}': {value}")

In [ ]:
# Look at the first announcement entry in detail
news_list = data.get("newsInfoLst", [])
print(f"Total announcements on this page: {len(news_list)}")
print()

if news_list:
    first = news_list[0]
    print("First announcement (raw JSON):")
    print(json.dumps(first, indent=2, ensure_ascii=False))

---
## Step 4: Understand the Data Fields

Each entry in `newsInfoLst` contains these key fields:

| Field | Type | Description | Example |
|-------|------|-------------|----------|
| `relTime` | string | Release date/time | `"11-12-2025 14:51"` |
| `stock` | array | Stock code(s) and name(s) | `[{"sc": "00005", "sn": "HSBC Holdings"}]` |
| `lTxt` | string | Category - Subcategory | `"Announcements and Notices - Inside Information"` |
| `title` | string | Announcement title | `"Interim Results for 2025"` |
| `webPath` | string | Path to the document (PDF) | `"/listedco/listconews/sehk/2025/..."` |
| `size` | string | Document file size | `"345KB"` |
| `lang` | string | Language | `"EN"` |

Let's parse a few entries into a clean format:

In [ ]:
# Parse the first 5 announcements into a readable format
for i, item in enumerate(news_list[:5], 1):
    # Extract stock info
    stocks = item.get("stock", [])
    stock_code = stocks[0].get("sc", "N/A") if stocks else "N/A"
    stock_name = stocks[0].get("sn", "N/A") if stocks else "N/A"
    
    # Parse category
    l_txt = item.get("lTxt", "")
    parts = l_txt.split(" - ", 1)
    category = parts[0].strip()
    subcategory = parts[1].strip() if len(parts) > 1 else ""
    
    print(f"--- Announcement {i} ---")
    print(f"  Stock:     [{stock_code}] {stock_name}")
    print(f"  Title:     {item.get('title', '')}")
    print(f"  Released:  {item.get('relTime', '')}")
    print(f"  Category:  {category}")
    print(f"  Detail:    {subcategory}")
    print(f"  Document:  https://www1.hkexnews.hk{item.get('webPath', '')}")
    print(f"  Size:      {item.get('size', '')}")
    print()

---
## Step 5: Fetch Multiple Pages

Each page contains ~20 announcements. To get more data, we fetch multiple pages with a small delay between requests to be respectful.

In [ ]:
import time

def fetch_hkex_announcements(pages=3, delay=1.0):
    """
    Fetch multiple pages of HKEX announcements.
    
    Args:
        pages: Number of pages to fetch
        delay: Seconds to wait between requests (be respectful!)
    
    Returns:
        List of parsed announcement dicts
    """
    all_announcements = []
    
    for page in range(1, pages + 1):
        url = HKEX_ENDPOINT.format(page=page)
        print(f"Fetching page {page}/{pages}...", end=" ")
        
        try:
            resp = requests.get(url, headers=headers, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            
            news_list = data.get("newsInfoLst", [])
            
            for item in news_list:
                stocks = item.get("stock", [])
                l_txt = item.get("lTxt", "")
                parts = l_txt.split(" - ", 1)
                
                announcement = {
                    "title": item.get("title", ""),
                    "stock_code": stocks[0].get("sc", "") if stocks else "",
                    "stock_name": stocks[0].get("sn", "") if stocks else "",
                    "released_at": item.get("relTime", ""),
                    "category": parts[0].strip() if parts else "",
                    "category_detail": parts[1].strip() if len(parts) > 1 else "",
                    "link": f"https://www1.hkexnews.hk{item.get('webPath', '')}",
                    "size": item.get("size", ""),
                }
                all_announcements.append(announcement)
            
            print(f"got {len(news_list)} announcements")
            
        except Exception as e:
            print(f"FAILED: {e}")
            break
        
        # Polite delay between requests
        if page < pages:
            time.sleep(delay)
    
    return all_announcements


# Fetch 3 pages of announcements
announcements = fetch_hkex_announcements(pages=3)
print(f"\nTotal announcements fetched: {len(announcements)}")

---
## Step 6: Explore the Data

Let's look at what categories and stock codes are in our dataset.

In [ ]:
# Count announcements by category
from collections import Counter

category_counts = Counter(a["category"] for a in announcements)

print("Announcements by Category:")
print("-" * 50)
for cat, count in category_counts.most_common():
    print(f"  {cat}: {count}")

In [ ]:
# Count announcements by stock code (top 15)
stock_counts = Counter(
    f"[{a['stock_code']}] {a['stock_name']}" 
    for a in announcements 
    if a['stock_code']
)

print("Top 15 Companies by Announcement Count:")
print("-" * 50)
for stock, count in stock_counts.most_common(15):
    print(f"  {stock}: {count}")

---
## Step 7: Filter for Biotech/Pharma Announcements

The HKEX JSON feed does **not** include industry/sector tags. There are two strategies:

1. **Filter by known stock codes** of biotech/pharma companies
2. **Filter by keywords** in announcement titles

### Strategy A: Filter by Biotech Stock Codes

In [ ]:
# Notable HKEX-listed biotech/pharma companies
# Source: Hang Seng Biotech Index, Chapter 18A listings
BIOTECH_STOCK_CODES = {
    "01177": "Sino Biopharmaceutical",
    "02269": "WuXi Biologics",
    "02359": "WuXi AppTec",
    "06160": "BeiGene",
    "09926": "Akeso",
    "09995": "RemeGen",
    "02162": "Keymed Biosciences",
    "06978": "Imeik Technology Development",
    "01801": "Innovent Biologics",
    "09969": "InnoCare Pharma",
    "09688": "Zai Lab",
    "09939": "Kintor Pharmaceutical",
    "03692": "Hansoh Pharmaceutical",
    "02696": "Hengrui Medicine (Shanghai Hengrui)",
    "06185": "CanSino Biologics",
    "03759": "SinoMab BioScience",
}

# Filter announcements
biotech_by_code = [
    a for a in announcements 
    if a["stock_code"] in BIOTECH_STOCK_CODES
]

print(f"Biotech/Pharma announcements (by stock code): {len(biotech_by_code)}")
print()
for a in biotech_by_code:
    print(f"  [{a['stock_code']}] {a['stock_name']}")
    print(f"    {a['title']}")
    print(f"    {a['released_at']} | {a['category']}")
    print()

### Strategy B: Filter by Keywords

In [ ]:
# Keywords relevant to biotech/pharma announcements
BIOTECH_KEYWORDS = [
    "clinical", "trial", "fda", "drug", "pharmaceutical",
    "biotech", "oncology", "therapy", "pipeline", "phase",
    "biologic", "vaccine", "antibody", "protein", "nmpa",
    "approval", "orphan", "indication", "efficacy",
]

# Filter announcements by title keywords (case-insensitive)
biotech_by_keyword = [
    a for a in announcements
    if any(kw in a["title"].lower() for kw in BIOTECH_KEYWORDS)
]

print(f"Biotech/Pharma announcements (by keyword): {len(biotech_by_keyword)}")
print()
for a in biotech_by_keyword:
    print(f"  [{a['stock_code']}] {a['stock_name']}")
    print(f"    {a['title']}")
    print(f"    {a['released_at']} | {a['category']}")
    print()

---
## Step 8: Filter by Category

You can also filter by announcement category. Useful categories:
- `"Announcements and Notices"` — General company announcements (includes Inside Information)
- `"Financial Statements"` — Results and financial reports
- `"Circulars"` — Shareholder circulars

The `category_detail` sub-field gives more specific types like `"Inside Information"`, `"Profit Warning"`, etc.

In [ ]:
# Filter for Inside Information disclosures (material non-public info)
inside_info = [
    a for a in announcements
    if "inside information" in a["category_detail"].lower()
]

print(f"Inside Information announcements: {len(inside_info)}")
print()
for a in inside_info[:10]:
    print(f"  [{a['stock_code']}] {a['stock_name']}")
    print(f"    {a['title']}")
    print(f"    {a['released_at']}")
    print()

In [ ]:
# Show all unique sub-categories found in the data
detail_counts = Counter(
    a["category_detail"] for a in announcements if a["category_detail"]
)

print("All Sub-Categories (category_detail):")
print("-" * 50)
for detail, count in detail_counts.most_common():
    print(f"  {detail}: {count}")

---
## Step 8.5: Getting English vs Chinese Versions

**IMPORTANT:** HKEX announcements can be in Chinese, English, or both. The document links use language suffixes:

| Suffix | Language |
|--------|----------|
| `_c.pdf` | Chinese (中文) |
| `_e.pdf` | English |

The API returns whichever version is listed first (usually Chinese). To get the English version:
1. Replace `_c.pdf` with `_e.pdf` in the link
2. Check if the English version exists (some announcements are Chinese-only)

Let's add a helper function to convert links:

In [ ]:
def get_english_link(link):
    """
    Convert a Chinese document link to English version.
    
    Args:
        link: Document URL (can be Chinese or English)
    
    Returns:
        English version URL
    """
    if "_c.pdf" in link:
        return link.replace("_c.pdf", "_e.pdf")
    elif "_c.htm" in link:
        return link.replace("_c.htm", "_e.htm")
    # Already English or no language suffix
    return link


def check_english_available(link, timeout=5):
    """
    Check if English version exists by making a HEAD request.
    
    Args:
        link: Document URL to check
        timeout: Request timeout in seconds
    
    Returns:
        True if document exists (HTTP 200), False otherwise
    """
    try:
        resp = requests.head(link, timeout=timeout, allow_redirects=True)
        return resp.status_code == 200
    except:
        return False


# Example: Convert biotech announcements to English
print("Converting Chinese links to English:")
print("=" * 70)

for a in biotech_by_code[:5]:
    chinese_link = a["link"]
    english_link = get_english_link(chinese_link)
    
    print(f"\n[{a['stock_code']}] {a['stock_name']}")
    print(f"  Title (Chinese): {a['title']}")
    print(f"  Chinese: {chinese_link}")
    print(f"  English: {english_link}")
    
    # Check if English version exists (optional - makes extra HTTP requests)
    # Uncomment if you want to verify availability:
    # exists = check_english_available(english_link)
    # print(f"  English available: {exists}")

### Enhanced Fetch with Both Languages

Let's update our fetch function to automatically include both Chinese and English links:

In [ ]:
def fetch_hkex_with_languages(pages=3, delay=1.0):
    """
    Fetch HKEX announcements with both Chinese and English links.
    
    Returns:
        List of announcements with 'link_chinese' and 'link_english' fields
    """
    all_announcements = []
    
    for page in range(1, pages + 1):
        url = HKEX_ENDPOINT.format(page=page)
        print(f"Fetching page {page}/{pages}...", end=" ")
        
        try:
            resp = requests.get(url, headers=headers, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            
            news_list = data.get("newsInfoLst", [])
            
            for item in news_list:
                stocks = item.get("stock", [])
                l_txt = item.get("lTxt", "")
                parts = l_txt.split(" - ", 1)
                
                web_path = item.get("webPath", "")
                full_link = f"https://www1.hkexnews.hk{web_path}"
                
                # Determine language and create both links
                if "_c.pdf" in full_link or "_c.htm" in full_link:
                    link_chinese = full_link
                    link_english = get_english_link(full_link)
                elif "_e.pdf" in full_link or "_e.htm" in full_link:
                    link_english = full_link
                    link_chinese = full_link.replace("_e.pdf", "_c.pdf").replace("_e.htm", "_c.htm")
                else:
                    # No language suffix - use as-is
                    link_chinese = full_link
                    link_english = full_link
                
                announcement = {
                    "title": item.get("title", ""),
                    "stock_code": stocks[0].get("sc", "") if stocks else "",
                    "stock_name": stocks[0].get("sn", "") if stocks else "",
                    "released_at": item.get("relTime", ""),
                    "category": parts[0].strip() if parts else "",
                    "category_detail": parts[1].strip() if len(parts) > 1 else "",
                    "link_chinese": link_chinese,
                    "link_english": link_english,
                    "size": item.get("size", ""),
                }
                all_announcements.append(announcement)
            
            print(f"got {len(news_list)} announcements")
            
        except Exception as e:
            print(f"FAILED: {e}")
            break
        
        if page < pages:
            time.sleep(delay)
    
    return all_announcements


# Fetch with both language links
announcements_bilingual = fetch_hkex_with_languages(pages=2)

print(f"\nTotal: {len(announcements_bilingual)} announcements")
print("\nExample (first 3):")
print("=" * 70)

for a in announcements_bilingual[:3]:
    print(f"\n[{a['stock_code']}] {a['stock_name']}")
    print(f"  Title: {a['title']}")
    print(f"  Released: {a['released_at']}")
    print(f"  Chinese: {a['link_chinese']}")
    print(f"  English: {a['link_english']}")

### Language Availability Notes

**Important caveats:**

1. **Not all announcements have English versions** — some companies only publish in Chinese
2. **The English link may 404** — always handle this gracefully in production code
3. **Title is always in the original language** — the JSON API doesn't provide translated titles
4. **Some documents use `.htm` instead of `.pdf`** — the same `_c` / `_e` suffix pattern applies

**Best practice for production:**
- Generate both links by default
- Try fetching the English version first
- Fall back to Chinese if English is unavailable (404)
- Log which language was actually retrieved

---
## Step 9: Save Results to JSON

Export filtered or unfiltered results for further processing in a data pipeline.

In [ ]:
# Save all announcements
output = {
    "fetched_at": datetime.utcnow().isoformat(),
    "source": "HKEXnews",
    "total_announcements": len(announcements),
    "announcements": announcements,
}

with open("hkex_all_announcements.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Saved {len(announcements)} announcements to hkex_all_announcements.json")

In [ ]:
# Save biotech-filtered announcements
biotech_all = biotech_by_code + [
    a for a in biotech_by_keyword if a not in biotech_by_code
]

biotech_output = {
    "fetched_at": datetime.utcnow().isoformat(),
    "source": "HKEXnews",
    "filter": "biotech/pharma (by stock code + keywords)",
    "total_announcements": len(biotech_all),
    "announcements": biotech_all,
}

with open("hkex_biotech_announcements.json", "w", encoding="utf-8") as f:
    json.dump(biotech_output, f, indent=2, ensure_ascii=False)

print(f"Saved {len(biotech_all)} biotech announcements to hkex_biotech_announcements.json")

---
## Step 10: Using the Parser Module

For production use, import the `HKEXNewsParser` class which wraps all of the above into a clean interface with retry logic.

In [ ]:
import sys
sys.path.insert(0, ".")

from hkex_parser import HKEXNewsParser

# Initialize
parser = HKEXNewsParser()

# Fetch 2 pages
announcements = parser.fetch_announcements(pages=2)
print(f"\nFetched {len(announcements)} announcements")

# Filter by keywords
biotech = parser.filter_by_keywords(announcements, ["clinical", "trial", "drug", "biotech"])
print(f"Biotech matches: {len(biotech)}")

# Filter by category
financial = parser.filter_by_category(announcements, ["Financial"])
print(f"Financial statements: {len(financial)}")

# Show results
for a in biotech[:3]:
    print(f"\n  [{a.stock_code}] {a.stock_name}")
    print(f"  {a.title}")
    print(f"  {a.released_at} | {a.category}")
    print(f"  {a.full_link}")

---
## Summary

### What We Learned

1. **HKEX has no official public API** for listed company announcements
2. The website uses an **internal JSON endpoint** that we can call directly
3. Each page returns ~20 announcements with stock code, title, category, and document link
4. **Filtering by sector** requires either known stock codes or keyword matching
5. Documents (PDFs) are accessible at `https://www1.hkexnews.hk{webPath}`
6. **Language handling**: Links use `_c.pdf` (Chinese) or `_e.pdf` (English) suffixes

### Key Endpoint

```
GET https://www1.hkexnews.hk/ncms/json/eds/lcisehk1relsdc_{page}.json
```

### Response Fields

| Field | Description |
|-------|-------------|
| `newsInfoLst` | Array of announcement objects |
| `relTime` | Release time (DD-MM-YYYY HH:mm) |
| `stock[].sc` | Stock code |
| `stock[].sn` | Stock name |
| `lTxt` | Category - Subcategory |
| `title` | Announcement title |
| `webPath` | Relative URL to document |
| `size` | Document file size |

### Language Suffixes

| Suffix | Language |
|--------|----------|
| `_c.pdf` or `_c.htm` | Chinese (中文) |
| `_e.pdf` or `_e.htm` | English |

**To get English versions:** Replace `_c` with `_e` in document links. Not all announcements have English versions.

### Caveats

- **Undocumented** — endpoint may change without notice
- **Rate limit** — add delays between requests (1-2 seconds)
- **Copyright** — HKEX data is copyrighted; review their terms of use
- **No sector filter** — must filter client-side by stock codes or keywords
- **Language availability** — English versions may not exist for all announcements
- **Titles not translated** — JSON API returns titles in original language only

### Finding Biotech Stock Codes

- **Chapter 18A "B" marker** in stock name = pre-revenue biotech
- **Hang Seng Biotech Index** — 30 largest biotech/pharma/medtech
- **HKEX Securities List**: [ListOfSecurities.xlsx](https://www.hkex.com.hk/eng/services/trading/securities/securitieslists/ListOfSecurities.xlsx)